# 10 — Engenharia de Features: razões e interações entre colunas antigas
## PRT Seguros

Ideia: em vez de só usar as colunas originais, criar **novas features que relacionam colunas
antigas** — razões (ex.: prêmio por apólice) e interações (ex.: engajamento × satisfação) costumam
capturar padrões que o modelo não vê olhando cada coluna isoladamente, mesmo em modelos de árvore.

Partimos do conjunto de features vencedor do notebook `09` (`sem_fracas_shift`, 68 colunas — já sem
as 4 features fracas/com shift) e testamos se adicionar features derivadas melhora o AUC-ROC de
validação o suficiente para justificar a complexidade extra.

## 1. Carregar dados e o conjunto de features selecionado no notebook 09

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready.csv")
val = pd.read_csv("dados_processados/val_model_ready.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready.csv")

with open("dados_processados/features_selecionadas.json") as f:
    selecao = json.load(f)
FEATURES_BASE = selecao["colunas"]
print(f"Conjunto base ({selecao['conjunto_escolhido']}): {len(FEATURES_BASE)} features")


Conjunto base (sem_fracas_shift): 68 features


## 2. Criar features derivadas

Todas com denominador protegido contra divisão por zero (`+ 1`). Aplicamos a mesma fórmula nas
três bases (treino, validação, Kaggle) para manter consistência.

In [2]:
def criar_features(df):
    df = df.copy()
    df["premio_por_cobertura"] = df["valor_premio_anual"] / (df["valor_cobertura_total"] + 1)
    df["franquia_pct_premio"] = df["franquia_media"] / (df["valor_premio_anual"] + 1)
    df["renda_por_dependente"] = df["renda_anual"] / (df["qtd_dependentes"] + 1)
    df["valor_imovel_por_renda"] = df["valor_imovel"] / (df["renda_anual"] + 1)
    df["apolices_por_produto"] = df["num_apolices_ativas"] / (df["num_produtos_contratados"] + 1)
    df["reclamacoes_por_sinistro"] = df["num_reclamacoes_12m"] / (df["num_sinistros_historico"] + 1)
    df["ligacoes_por_sinistro"] = df["num_ligacoes_suporte_12m"] / (df["num_sinistros_historico"] + 1)
    df["engajamento_x_satisfacao"] = df["score_engajamento_digital"] * df["satisfacao_nps"]
    df["desconto_x_tempo_cliente"] = df["desconto_aplicado_pct"] * df["tempo_cliente_dias"]
    df["relacionamento_por_ano_cliente"] = df["indice_relacionamento"] / (df["tempo_cliente_dias"] / 365 + 1)
    df["contato_menos_login"] = df["dias_ultimo_contato"] - df["ultimo_login_portal_dias"]
    return df

FEATURES_NOVAS = [
    "premio_por_cobertura", "franquia_pct_premio", "renda_por_dependente",
    "valor_imovel_por_renda", "apolices_por_produto", "reclamacoes_por_sinistro",
    "ligacoes_por_sinistro", "engajamento_x_satisfacao", "desconto_x_tempo_cliente",
    "relacionamento_por_ano_cliente", "contato_menos_login",
]

train_fe = criar_features(train)
val_fe = criar_features(val)
kaggle_fe = criar_features(kaggle)

print(f"Novas features criadas: {len(FEATURES_NOVAS)}")
train_fe[FEATURES_NOVAS].describe().T[["mean", "std", "min", "max"]]


Novas features criadas: 11


,mean,std,min,max
premio_por_cobertura,0.009807,0.004635,0.000620,0.116747
franquia_pct_premio,0.789945,0.236405,0.133474,2.741235
renda_por_dependente,49174.586427,37702.380231,2520.000000,668500.000000
valor_imovel_por_renda,4.540038,2.758279,0.183583,63.556629
apolices_por_produto,0.971520,0.717726,0.125000,3.000000
reclamacoes_por_sinistro,0.589685,0.635333,0.000000,2.500000
ligacoes_por_sinistro,1.330758,1.105789,0.000000,6.000000
engajamento_x_satisfacao,316.502856,163.864880,0.000000,1075.000000
desconto_x_tempo_cliente,187.302716,130.981833,0.000000,1092.769000
relacionamento_por_ano_cliente,10.673676,9.789579,0.568843,84.458228


## 3. Comparar CatBoost: features base vs. base + engenharia

Mesmos hiperparâmetros do `05_catboost_proba.ipynb` — a única variável é o conjunto de colunas de entrada.

In [3]:
def avaliar(cols, nome):
    X_train, y_train = train_fe[cols], train_fe[TARGET_COL]
    X_val, y_val = val_fe[cols], val_fe[TARGET_COL]

    X_tr_es, X_es, y_tr_es, y_es = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
    )
    neg_pos_ratio = (y_tr_es == 0).sum() / (y_tr_es == 1).sum()

    modelo = CatBoostClassifier(
        iterations=2000, depth=6, learning_rate=0.03,
        scale_pos_weight=neg_pos_ratio, eval_metric="AUC",
        random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
    )
    modelo.fit(X_tr_es, y_tr_es, eval_set=(X_es, y_es))
    proba_val = modelo.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, proba_val)
    print(f"{nome:<30} ({len(cols):>2} features) -> AUC-ROC = {auc:.4f}")
    return auc, modelo

auc_base, _ = avaliar(FEATURES_BASE, "base (sem engenharia)")
auc_fe, modelo_fe = avaliar(FEATURES_BASE + FEATURES_NOVAS, "base + features derivadas")

print(f"\nGanho da engenharia de features: {auc_fe - auc_base:+.4f}")


base (sem engenharia)          (68 features) -> AUC-ROC = 0.8260


base + features derivadas      (79 features) -> AUC-ROC = 0.8259

Ganho da engenharia de features: -0.0001


## 4. Quais features novas o modelo mais usou?

Se as derivadas aparecerem bem posicionadas na *feature importance*, isso reforça que elas carregam
sinal que as colunas originais sozinhas não davam tão diretamente.

In [4]:
importancias = pd.Series(
    modelo_fe.get_feature_importance(), index=FEATURES_BASE + FEATURES_NOVAS
).sort_values(ascending=False)

print("Posição de cada feature nova no ranking geral de importância:")
ranking = importancias.rank(ascending=False).astype(int)
for f in FEATURES_NOVAS:
    print(f"  {f:<32} posição {ranking[f]:>3} de {len(importancias)}  (importância={importancias[f]:.2f})")


Posição de cada feature nova no ranking geral de importância:
  premio_por_cobertura             posição  12 de 79  (importância=1.68)
  franquia_pct_premio              posição  25 de 79  (importância=0.78)
  renda_por_dependente             posição  31 de 79  (importância=0.61)
  valor_imovel_por_renda           posição  24 de 79  (importância=0.79)
  apolices_por_produto             posição   2 de 79  (importância=10.50)
  reclamacoes_por_sinistro         posição  33 de 79  (importância=0.59)
  ligacoes_por_sinistro            posição  29 de 79  (importância=0.68)
  engajamento_x_satisfacao         posição  17 de 79  (importância=1.14)
  desconto_x_tempo_cliente         posição   8 de 79  (importância=2.72)
  relacionamento_por_ano_cliente   posição  13 de 79  (importância=1.55)
  contato_menos_login              posição  28 de 79  (importância=0.70)


## 5. Decidir o conjunto final e salvar as bases `_v2`

Se `auc_fe > auc_base`, mantemos as features novas. Caso contrário, seguimos só com o conjunto base
do notebook `09`. De qualquer forma, salvamos `train_model_ready_v2.csv`, `val_model_ready_v2.csv`
e `kaggle_model_ready_v2.csv` com o conjunto vencedor — esses arquivos alimentam o próximo notebook
de tuning de hiperparâmetro.

In [5]:
usar_features_novas = auc_fe >= auc_base
FEATURES_FINAIS = FEATURES_BASE + FEATURES_NOVAS if usar_features_novas else FEATURES_BASE

print(f"Features novas ajudaram? {'Sim' if usar_features_novas else 'Não'}")
print(f"Conjunto final: {len(FEATURES_FINAIS)} features")

colunas_salvar = FEATURES_FINAIS + [ID_COL]

train_fe[colunas_salvar + [TARGET_COL]].to_csv("dados_processados/train_model_ready_v2.csv", index=False)
val_fe[colunas_salvar + [TARGET_COL]].to_csv("dados_processados/val_model_ready_v2.csv", index=False)
kaggle_fe[colunas_salvar].to_csv("dados_processados/kaggle_model_ready_v2.csv", index=False)

with open("dados_processados/features_selecionadas_v2.json", "w") as f:
    json.dump({"usa_features_novas": usar_features_novas, "colunas": FEATURES_FINAIS}, f, indent=2)

print("Arquivos _v2 salvos em dados_processados/")


Features novas ajudaram? Não
Conjunto final: 68 features


Arquivos _v2 salvos em dados_processados/
